# 03 — Graph Neural Network (per-target)

Phase 3 of **CRC_Inhibitor_ML**. The actual model. Trains one **Graph Isomorphism Network (GIN)** per target, evaluates on the same scaffold split as Phase 2, and compares head-to-head against the RF baseline.

Workflow:

1. **Load** the clean CSV from Phase 1
2. **Featurize** each SMILES as a PyTorch Geometric molecular graph — atoms become nodes with chemical features (element, charge, hybridization, aromaticity, ring membership, etc.), bonds become edges with bond-type features
3. **Split** each target into train / val / test using the same scaffold-split logic as Phase 2, then random-split the train pool to carve off a val set for early stopping
4. **Train** a GIN regressor per target (3 message-passing layers, hidden_dim=128, dropout=0.2, Adam optimizer, MSE loss, early stopping on val RMSE)
5. **Evaluate** on the held-out test set — same scaffold split as Phase 2 so the comparison is direct
6. **Compare** vs Phase 2 baselines (load `reports/baseline_rf_metrics.json`, show side-by-side table)
7. **Save** trained model checkpoints to `models/` and metrics to `reports/gnn_single_target_metrics.json`

**Why GIN:** Graph Isomorphism Networks are provably as expressive as the Weisfeiler-Lehman graph isomorphism test, which makes them a strong default for molecular property prediction. Simpler than AttentiveFP / D-MPNN but typically within a few R² points of them on standard benchmarks. Phase 3.5 (if we want it) could swap in a chemistry-specific architecture for a possible extra bump.

Expect training time: ~30 sec to 5 min per target on the RTX 3060 depending on size.

In [ ]:
from pathlib import Path
from collections import defaultdict
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_add_pool

from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

RDLogger.DisableLog("rdApp.*")

PROJECT_ROOT  = Path(r"C:\Users\palla\OneDrive\Documents\Coding Projects\CRC_Inhibitor_ML")
CLEAN_CSV     = PROJECT_ROOT / "data" / "interim"   / "chembl_crc_targets_clean.csv"
MODELS_DIR    = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR   = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
GNN_METRICS_PATH = REPORTS_DIR / "gnn_single_target_metrics.json"
RF_METRICS_PATH  = REPORTS_DIR / "baseline_rf_metrics.json"

TARGETS = {
    "CHEMBL2189121": "KRAS",
    "CHEMBL5145":    "BRAF",
    "CHEMBL203":     "EGFR",
    "CHEMBL4005":    "PIK3CA",
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU:    {torch.cuda.get_device_name(0)}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## Cell 1 — load clean data + recompute scaffolds

Same dataset as Phase 2. We recompute Murcko scaffolds here (rather than importing them) so this notebook is self-contained.

In [ ]:
clean = pd.read_csv(CLEAN_CSV)
print(f"Loaded {len(clean):,} rows")

# Parse SMILES once, reuse for featurization + scaffold computation
print("\nParsing SMILES...")
mols = [Chem.MolFromSmiles(s) for s in tqdm(clean["canonical_smiles_std"])]
valid = np.array([m is not None for m in mols])
clean = clean[valid].reset_index(drop=True)
mols  = [m for m, ok in zip(mols, valid) if ok]

print("Computing Murcko scaffolds...")
scaffolds = []
for m in tqdm(mols):
    try:
        scaffolds.append(MurckoScaffold.MurckoScaffoldSmiles(mol=m, includeChirality=False))
    except Exception:
        scaffolds.append("")
clean["scaffold"] = scaffolds

print(f"\nReady: {len(clean):,} molecules, {clean['scaffold'].nunique():,} unique scaffolds")

## Cell 2 — atom & bond featurization

Each molecule becomes a `torch_geometric.data.Data` object with:

- `x`: node feature matrix — one row per atom, columns are concatenated one-hot encodings of (element, hybridization) plus numeric features (formal charge, total H count, degree, aromaticity flag, in-ring flag). 24 features per atom.
- `edge_index`: 2 × (2 × num_bonds) tensor of [source_atom_idx, target_atom_idx] pairs. Each bond becomes two directed edges (one each way) so message passing is symmetric.
- `edge_attr`: edge feature matrix — bond type one-hot + conjugation + in-ring flags. 6 features per edge.
- `y`: the pIC50 label.

Standard cheminformatics featurization — nothing exotic.

In [ ]:
ATOM_TYPES = ["C", "N", "O", "F", "P", "S", "Cl", "Br", "I", "B", "Si", "H", "Other"]
HYB_TYPES = [
    Chem.HybridizationType.S,
    Chem.HybridizationType.SP,
    Chem.HybridizationType.SP2,
    Chem.HybridizationType.SP3,
    Chem.HybridizationType.SP3D,
    Chem.HybridizationType.SP3D2,
]
BOND_TYPES = [
    Chem.BondType.SINGLE,
    Chem.BondType.DOUBLE,
    Chem.BondType.TRIPLE,
    Chem.BondType.AROMATIC,
]
N_ATOM_FEATURES = len(ATOM_TYPES) + len(HYB_TYPES) + 5
N_BOND_FEATURES = len(BOND_TYPES) + 2

def one_hot(value, options):
    """Return one-hot encoding; if value not in options, last slot is 'Other'."""
    return [int(value == o) for o in options]

def atom_features(atom):
    sym = atom.GetSymbol() if atom.GetSymbol() in ATOM_TYPES else "Other"
    return (
        one_hot(sym, ATOM_TYPES)
        + one_hot(atom.GetHybridization(), HYB_TYPES)
        + [
            atom.GetFormalCharge(),
            atom.GetTotalNumHs(),
            atom.GetDegree(),
            int(atom.GetIsAromatic()),
            int(atom.IsInRing()),
        ]
    )

def bond_features(bond):
    return (
        one_hot(bond.GetBondType(), BOND_TYPES)
        + [int(bond.GetIsConjugated()), int(bond.IsInRing())]
    )

def mol_to_pyg(mol, y_value):
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)

    src, dst, eattr = [], [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf = bond_features(bond)
        src += [i, j]
        dst += [j, i]
        eattr += [bf, bf]

    if len(src) == 0:
        # Single-atom molecule (very rare in this dataset)
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr  = torch.zeros((0, N_BOND_FEATURES), dtype=torch.float)
    else:
        edge_index = torch.tensor([src, dst], dtype=torch.long)
        edge_attr  = torch.tensor(eattr, dtype=torch.float)

    y = torch.tensor([y_value], dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)

print(f"N_ATOM_FEATURES = {N_ATOM_FEATURES}")
print(f"N_BOND_FEATURES = {N_BOND_FEATURES}")

# Featurize ALL molecules once — keep aligned with `clean`
print("\nFeaturizing molecules into PyG Data objects...")
pyg_data = []
for mol, y in tqdm(list(zip(mols, clean["pic50"].values)), total=len(mols)):
    pyg_data.append(mol_to_pyg(mol, float(y)))
print(f"Built {len(pyg_data):,} graphs")

## Cell 3 — scaffold split + train/val split

Same scaffold split function as Phase 2 — 80% train_val pool / 20% test, where scaffolds never appear in both. Then we further random-split the train_val pool 80/20 into actual train and val sets. Val is used to early-stop training; test is touched only at the very end for the final reported metric.

Why this matters: if we early-stop on the test set, we leak test information into model selection. Using a held-out val set keeps the final test number honest.

In [ ]:
def scaffold_split_indices(scaffolds, frac_train=0.8):
    """Bemis-Murcko scaffold split. Deterministic given input order."""
    groups = defaultdict(list)
    for i, s in enumerate(scaffolds):
        groups[s].append(i)
    sorted_groups = sorted(groups.values(), key=len, reverse=True)
    cutoff = int(frac_train * len(scaffolds))
    train, test = [], []
    for indices in sorted_groups:
        if len(train) + len(indices) <= cutoff:
            train.extend(indices)
        else:
            test.extend(indices)
    return np.array(train), np.array(test)

def split_target(target_id, frac_val=0.2):
    """Per-target: scaffold-split to (train_val, test), then random-split train_val to (train, val)."""
    mask = (clean["target_chembl_id"] == target_id).values
    target_indices = np.where(mask)[0]
    target_scaffolds = clean.loc[mask, "scaffold"].tolist()

    # Scaffold-split (indices are LOCAL to this target's slice)
    tv_local, te_local = scaffold_split_indices(target_scaffolds, frac_train=0.8)

    # Random-split train_val into train + val
    rng = np.random.RandomState(SEED)
    shuffled = rng.permutation(tv_local)
    n_val = int(frac_val * len(shuffled))
    val_local   = shuffled[:n_val]
    train_local = shuffled[n_val:]

    # Translate local indices back to global pyg_data indices
    return (
        target_indices[train_local],
        target_indices[val_local],
        target_indices[te_local],
    )

## Cell 4 — GIN model

**Graph Isomorphism Network** with 3 message-passing layers.

Architecture:

1. **Input projection** — linear layer maps the 24-dim atom features → 128-dim hidden representation
2. **3 × GINConv layers** — each does message passing: every atom updates its representation by summing its neighbors' representations through a small MLP. After 3 rounds, each atom "knows about" its 3-hop neighborhood.
3. **Global add pool** — sum all atom embeddings to get one molecule-level vector
4. **Predictor MLP** — 2-layer feedforward with dropout → single pIC50 prediction

Why sum-pool instead of mean-pool: sum preserves info about molecule size (a 50-atom drug looks different from a 10-atom fragment). For property prediction, sum is the standard choice.

In [ ]:
class GINRegressor(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, n_layers=3, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(in_dim, hidden_dim)

        self.convs = nn.ModuleList()
        for _ in range(n_layers):
            mlp = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
            )
            self.convs.append(GINConv(mlp))

        self.dropout = nn.Dropout(dropout)
        self.predictor = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        h = self.input_proj(x)
        for conv in self.convs:
            h = conv(h, edge_index)
            h = F.relu(h)
            h = self.dropout(h)
        g = global_add_pool(h, batch)
        return self.predictor(g).squeeze(-1)

## Cell 5 — train loop

Standard PyTorch training loop with early stopping. Trains for up to 200 epochs, stops if val RMSE doesn't improve for 20 epochs.

In [ ]:
def evaluate_model(model, loader):
    model.eval()
    preds_all, y_all = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            pred = model(batch).cpu().numpy()
            preds_all.append(pred)
            y_all.append(batch.y.cpu().numpy())
    preds = np.concatenate(preds_all)
    y     = np.concatenate(y_all)
    return {
        "r2":   float(r2_score(y, preds)),
        "rmse": float(np.sqrt(mean_squared_error(y, preds))),
        "mae": float(mean_absolute_error(y, preds)),
    }

def train_target(train_data, val_data, test_data, target_name,
                 epochs=200, lr=1e-3, batch_size=64, patience=20):
    model = GINRegressor(in_dim=N_ATOM_FEATURES).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn   = nn.MSELoss()

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_data,   batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_data,  batch_size=batch_size, shuffle=False)

    best_val_rmse = float("inf")
    best_state    = None
    epochs_no_improve = 0
    train_history = []

    pbar = tqdm(range(epochs), desc=f"{target_name:7s}")
    for epoch in pbar:
        model.train()
        running_loss = 0.0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            pred = model(batch)
            loss = loss_fn(pred, batch.y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * batch.num_graphs
        train_loss = running_loss / len(train_data)

        val_metrics = evaluate_model(model, val_loader)
        train_history.append({"epoch": epoch, "train_mse": train_loss, "val_rmse": val_metrics["rmse"]})

        if val_metrics["rmse"] < best_val_rmse:
            best_val_rmse = val_metrics["rmse"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        pbar.set_postfix({
            "train_mse": f"{train_loss:.3f}",
            "val_rmse":  f"{val_metrics['rmse']:.3f}",
            "best_val":  f"{best_val_rmse:.3f}",
        })

        if epochs_no_improve >= patience:
            break

    # Restore best-val model and evaluate on TEST (the real number)
    model.load_state_dict(best_state)
    test_metrics = evaluate_model(model, test_loader)
    test_metrics["n_train"]  = len(train_data)
    test_metrics["n_val"]    = len(val_data)
    test_metrics["n_test"]   = len(test_data)
    test_metrics["n_epochs"] = epoch + 1
    test_metrics["best_val_rmse"] = float(best_val_rmse)
    return model, test_metrics, train_history

## Cell 6 — train per-target

Loop through all four targets, train a fresh model on each, collect test metrics. Each model also gets saved to `models/gin_<target>.pt` so Phase 4 (or future inference) can load them.

In [ ]:
all_results = []

for target_id, target_name in TARGETS.items():
    print(f"\n{'='*60}")
    print(f" {target_name} ({target_id})")
    print(f"{'='*60}")

    train_idx, val_idx, test_idx = split_target(target_id)
    train_data = [pyg_data[i] for i in train_idx]
    val_data   = [pyg_data[i] for i in val_idx]
    test_data  = [pyg_data[i] for i in test_idx]
    print(f"Train: {len(train_data):,}  Val: {len(val_data):,}  Test: {len(test_data):,}")

    if len(train_data) < 100:
        print(f"Skipping {target_name} — too few training molecules.")
        continue

    model, metrics, history = train_target(train_data, val_data, test_data, target_name)
    metrics["target"] = target_name
    all_results.append(metrics)

    # Save model
    ckpt_path = MODELS_DIR / f"gin_{target_name.lower()}.pt"
    torch.save({
        "state_dict": model.state_dict(),
        "in_dim":     N_ATOM_FEATURES,
        "target":     target_name,
        "target_chembl_id": target_id,
        "metrics":    metrics,
    }, ckpt_path)
    print(f"Final TEST metrics: R\u00b2={metrics['r2']:.3f}  RMSE={metrics['rmse']:.3f}  MAE={metrics['mae']:.3f}")
    print(f"Saved checkpoint → {ckpt_path.name}")

results_df = pd.DataFrame(all_results)

## Cell 7 — head-to-head: GNN vs RF baseline

Load Phase 2's saved metrics, align by target + scaffold split, print side-by-side. The column we care about is `Δ R²` (GNN minus RF on scaffold split). Positive = GNN improvement; negative = something's wrong.

In [ ]:
with open(RF_METRICS_PATH) as f:
    rf_results = json.load(f)
rf_df = pd.DataFrame(rf_results)
rf_scaffold = rf_df[rf_df["split"] == "scaffold"].set_index("target")[["r2", "rmse", "mae"]]
rf_scaffold.columns = ["r2_rf", "rmse_rf", "mae_rf"]

gnn_df = results_df.set_index("target")[["r2", "rmse", "mae"]]
gnn_df.columns = ["r2_gnn", "rmse_gnn", "mae_gnn"]

compare = rf_scaffold.join(gnn_df, how="inner")
compare["r2_delta"]   = compare["r2_gnn"]   - compare["r2_rf"]
compare["rmse_delta"] = compare["rmse_gnn"] - compare["rmse_rf"]

compare = compare[[
    "r2_rf", "r2_gnn", "r2_delta",
    "rmse_rf", "rmse_gnn", "rmse_delta",
]]
print("Scaffold-split comparison (the realistic number):\n")
print(compare.round(3))

## Cell 8 — save GNN metrics

Bundle the per-target test metrics into `reports/gnn_single_target_metrics.json` so Phase 4 (multi-target ESM-2 model) can compare against both this and the RF baseline.

In [ ]:
with open(GNN_METRICS_PATH, "w") as f:
    json.dump(all_results, f, indent=2)

print(f"Saved {len(all_results)} GNN metric records to {GNN_METRICS_PATH}")
print()
print("Contents:")
print(json.dumps(all_results, indent=2))